In [1]:
# ============================================================
# EXPERIMENT 001
# XGBoost baseline for Playground Series S6E8
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

# Make paths work whether the notebook is run from the project
# root or from the notebooks folder.
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

print("\nTrain columns:")
print(train.columns.tolist())

assert TARGET in train.columns, f"{TARGET!r} is missing from train.csv"
assert TARGET not in test.columns, f"{TARGET!r} unexpectedly appears in test.csv"
assert len(test) == len(sample_submission), (
    "Test and sample-submission row counts do not match."
)


# ============================================================
# 3. BASIC DATA AUDIT
# ============================================================

print("\nTarget counts:")
print(train[TARGET].value_counts(dropna=False))

print("\nTarget proportions:")
print(
    train[TARGET]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
)

print("\nData types:")
print(train.dtypes.value_counts())

missing_summary = (
    train.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percent"] = (
    100 * missing_summary["missing_count"] / len(train)
)

print("\nColumns with missing values:")
print(missing_summary[missing_summary["missing_count"] > 0].head(30))

duplicate_count = train.duplicated().sum()
print(f"\nFully duplicated training rows: {duplicate_count:,}")


# ============================================================
# 4. CREATE FEATURES AND TARGET
# ============================================================

columns_to_drop = [TARGET]

if ID_COLUMN in train.columns:
    columns_to_drop.append(ID_COLUMN)

X = train.drop(columns=columns_to_drop)
y = train[TARGET].astype(int)

test_columns_to_drop = []

if ID_COLUMN in test.columns:
    test_columns_to_drop.append(ID_COLUMN)

X_test = test.drop(columns=test_columns_to_drop)

assert list(X.columns) == list(X_test.columns), (
    "Train and test feature columns are not aligned."
)

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_columns = X.columns.difference(categorical_columns).tolist()

print(f"\nNumber of model features: {X.shape[1]}")
print(f"Numerical features:       {len(numeric_columns)}")
print(f"Categorical features:     {len(categorical_columns)}")

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 5. PREPROCESSING
# ============================================================

# Compatibility across recent and older scikit-learn versions.
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", one_hot_encoder)
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ],
    remainder="drop"
)


# ============================================================
# 6. XGBOOST CONFIGURATION
# ============================================================

model_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": 1500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "early_stopping_rounds": 75,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# ============================================================
# 7. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

out_of_fold_predictions = np.zeros(len(train), dtype=float)
test_predictions = np.zeros(len(test), dtype=float)

fold_scores = []
best_iterations = []

experiment_start = time()

for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(X, y),
    start=1
):
    fold_start = time()

    X_train_fold = X.iloc[train_indices]
    X_validation_fold = X.iloc[validation_indices]

    y_train_fold = y.iloc[train_indices]
    y_validation_fold = y.iloc[validation_indices]

    # Fit preprocessing only on the current training fold.
    fold_preprocessor = clone(preprocessor)

    X_train_processed = fold_preprocessor.fit_transform(X_train_fold)
    X_validation_processed = fold_preprocessor.transform(
        X_validation_fold
    )
    X_test_processed = fold_preprocessor.transform(X_test)

    model = XGBClassifier(**model_parameters)

    model.fit(
        X_train_processed,
        y_train_fold,
        eval_set=[
            (X_validation_processed, y_validation_fold)
        ],
        verbose=False
    )

    validation_probabilities = model.predict_proba(
        X_validation_processed
    )[:, 1]

    fold_test_probabilities = model.predict_proba(
        X_test_processed
    )[:, 1]

    out_of_fold_predictions[validation_indices] = (
        validation_probabilities
    )

    test_predictions += fold_test_probabilities / N_SPLITS

    fold_auc = roc_auc_score(
        y_validation_fold,
        validation_probabilities
    )

    fold_scores.append(fold_auc)

    best_iteration = getattr(model, "best_iteration", None)
    best_iterations.append(best_iteration)

    fold_minutes = (time() - fold_start) / 60

    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"AUC: {fold_auc:.6f} | "
        f"Best iteration: {best_iteration} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 8. VALIDATION RESULTS
# ============================================================

overall_oof_auc = roc_auc_score(
    y,
    out_of_fold_predictions
)

mean_fold_auc = np.mean(fold_scores)
std_fold_auc = np.std(fold_scores)

total_minutes = (time() - experiment_start) / 60

print("\n" + "=" * 60)
print("EXPERIMENT 001 RESULTS")
print("=" * 60)

for fold_number, score in enumerate(fold_scores, start=1):
    print(f"Fold {fold_number} AUC: {score:.6f}")

print(f"\nMean fold AUC: {mean_fold_auc:.6f}")
print(f"Fold AUC SD:   {std_fold_auc:.6f}")
print(f"OOF AUC:       {overall_oof_auc:.6f}")
print(f"Runtime:       {total_minutes:.2f} minutes")


# ============================================================
# 9. SANITY CHECK PREDICTIONS
# ============================================================

print("\nTest prediction summary:")
print(pd.Series(test_predictions).describe())

assert np.isfinite(test_predictions).all(), (
    "Submission contains non-finite predictions."
)

assert ((test_predictions >= 0) & (test_predictions <= 1)).all(), (
    "Predictions must be probabilities between zero and one."
)

if np.std(test_predictions) == 0:
    raise ValueError(
        "All test predictions are identical. Something went wrong."
    )


# ============================================================
# 10. CREATE SUBMISSION
# ============================================================

submission = sample_submission.copy()

if TARGET not in submission.columns:
    raise KeyError(
        f"{TARGET!r} is not present in sample_submission.csv. "
        f"Available columns: {submission.columns.tolist()}"
    )

submission[TARGET] = test_predictions

submission_path = (
    SUBMISSION_DIR /
    "exp_001_xgb_baseline.csv"
)

submission.to_csv(submission_path, index=False)

print(f"\nSubmission saved to:\n{submission_path}")
print("\nSubmission preview:")
print(submission.head())

print("\nSubmission columns:")
print(submission.columns.tolist())

Train shape:             (691369, 14)
Test shape:              (296302, 13)
Sample submission shape: (296302, 2)

Train columns:
['id', 'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'addicted_label']

Target counts:
addicted_label
1    490474
0    200895
Name: count, dtype: int64

Target proportions:
addicted_label
0    0.290576
1    0.709424
Name: proportion, dtype: float64

Data types:
float64    9
object     3
int64      2
Name: count, dtype: int64

Columns with missing values:
                         missing_count  missing_percent
social_media_hours              133995        19.381112
gaming_hours                    126821        18.343461
weekend_screen_time             112063        16.208855
daily_screen_time_hours          95854        13.864376
app_opens_per_day                80710        11.67394

### Experiment Log

In [2]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

In [3]:
experiments = log_experiment(
    experiment_id="EXP-001",
    description=(
        "Initial XGBoost baseline using median-imputed numerical features "
        "and most-frequent-imputed, one-hot-encoded categorical features."
    ),
    model="XGBClassifier",
    features=X.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.962477,
        0.963382,
        0.963244
    ],
    kaggle_score=0.96449,
    changes=(
        "Established the first end-to-end baseline using all 12 available "
        "features, fold-specific preprocessing, early stopping, and averaged "
        "test predictions."
    ),
    submission_file="exp_001_xgb_baseline.csv",
    notes=(
        "OOF AUC was 0.963034 with fold SD 0.000398. Kaggle exceeded OOF "
        "by 0.001456. Best iterations were 1499, 1498, and 1499."
    )
)

experiments

Logged EXP-001
CV mean:       0.963034
CV SD:         0.000398
Kaggle score:  0.964490
Kaggle-CV gap: +0.001456
Log saved to:  C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\experiment_log.csv


,experiment_id,timestamp,description,model,features,n_features,validation_method,cv_scores,cv_mean,cv_std,kaggle_score,kaggle_cv_gap,changes,submission_file,notes
0,EXP-001,2026-08-02 21:49:55,Initial XGBoost baseline using median-imputed ...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.962477, 0.963382, 0.963244]",0.963034,0.000398,0.96449,0.001456,Established the first end-to-end baseline usin...,exp_001_xgb_baseline.csv,OOF AUC was 0.963034 with fold SD 0.000398. Ka...
